In [ ]:
# YOLO format
# <class_id> <x_center> <y_center> <width> <height>

In [ ]:
# Function to convert COCO bbox to YOLO format
def coco_to_yolo_bbox(bbox, img_width, img_height):
    x, y, w, h = bbox
    x_center = (x + w / 2) / img_width
    y_center = (y + h / 2) / img_height
    width = w / img_width
    height = h / img_height
    return [x_center, y_center, width, height]

# Convert all annotations to YOLO format and save them
for img_info in coco_data["images"]:
    img_id = img_info["id"]
    img_name = img_info["file_name"]
    img_width, img_height = new_width, new_height  # Use resized dimensions

    # Get annotations for this image
    annotations = [ann for ann in coco_data["annotations"] if ann["image_id"] == img_id]

    # Prepare YOLO format annotations
    yolo_annotations = []
    for ann in annotations:
        class_id = ann["category_id"]
        bbox = ann["bbox"]
        yolo_bbox = coco_to_yolo_bbox(bbox, img_width, img_height)
        yolo_annotations.append(f"{class_id} {' '.join(map(str, yolo_bbox))}")

    # Save YOLO annotations to a .txt file
    label_path = os.path.join(processed_folder, img_name.replace(".jpg", ".txt"))
    with open(label_path, "w") as f:
        f.write("\n".join(yolo_annotations))

print("✅ COCO annotations converted to YOLO format and saved.")

In [ ]:
import shutil
from sklearn.model_selection import train_test_split

# Create train and val directories
os.makedirs("dataset/images/train", exist_ok=True)
os.makedirs("dataset/images/val", exist_ok=True)
os.makedirs("dataset/labels/train", exist_ok=True)
os.makedirs("dataset/labels/val", exist_ok=True)

# Split images and labels into train and val sets
image_files = os.listdir(processed_folder)
train_files, val_files = train_test_split(image_files, test_size=0.2, random_state=42)

# Move files to train and val directories
for file in train_files:
    if file.endswith(".jpg"):
        shutil.move(os.path.join(processed_folder, file), os.path.join("dataset/images/train", file))
        shutil.move(os.path.join(processed_folder, file.replace(".jpg", ".txt")), os.path.join("dataset/labels/train", file.replace(".jpg", ".txt")))
for file in val_files:
    if file.endswith(".jpg"):
        shutil.move(os.path.join(processed_folder, file), os.path.join("dataset/images/val", file))
        shutil.move(os.path.join(processed_folder, file.replace(".jpg", ".txt")), os.path.join("dataset/labels/val", file.replace(".jpg", ".txt")))

print("✅ Dataset split into train and val sets.")